# supra50m → Mercury-2 diffusion LM **by the translator C alone** (run 3: self-zoo)

**Contract.** Model B is *never trained*. `B* = C(supra50m)` = the donor's own weights +
per-block corrections **emitted by C**, + an emitted [MASK] embedding. C is a per-block,
weight-tied hypernet trained through the diffusion loss on a **zoo of small AR Llamas**,
then applied **zero-shot** to the 12-layer donor. All data is **self-generated by supra**.

**Run-1** failed on output-frame mismatch → fixed by **SVD-frame covariant deltas**
`ΔW = Uᵣ · A(z) · Vᵣᵀ`. **Run-2** failed on the **donor-distribution gap**: barely-trained
shallow zoo donors look nothing like a trained model, so supra's signatures were OOD and the
learned corrections wrong (C degenerated to a global smoothing delta). **Run-3** closes both:
the zoo is **supra's own depth-truncated sub-stacks** (layers 0..L−1 + final norm + tied
head, L∈{2,4,6,8,10} — valid AR models). Zoo frames and weight statistics are *exactly* the
target's; C learns the per-block AR→denoiser rule on real supra blocks in shallow contexts
and extrapolates to L=12. *Disclosure:* the zoo shares weights with the target — held-out is
the full-depth composition and blocks 10–11 (never seen in training). B is never trained.

**Setup.** GPU + Internet, Run All. Saves `B*` to `/kaggle/working`.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import glob, json, math, time, torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import load_file, save_file

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
MODEL_ID = 'SupraLabs/Supra-50M-Instruct'

def resolve_model():
    env = os.environ.get('CKPT_DIR')
    if env and os.path.exists(os.path.join(env, 'config.json')): return env
    ds = glob.glob('/kaggle/input/**/config.json', recursive=True)
    if ds: return os.path.dirname(ds[0])
    from huggingface_hub import snapshot_download
    return snapshot_download(MODEL_ID, allow_patterns=['config.json', 'model.safetensors', 'tokenizer.json'])

MODEL_DIR = resolve_model()
N_GEN     = 1024
N_HELD    = 64
SEQ_LEN   = 256
TRUNC_DEPTHS = [2, 4, 6, 8, 10]   # self-zoo: supra's own sub-stacks (free, in-frame)
C_STEPS   = 3000
C_BS      = 8
RANK      = 24          # SVD-frame correction rank (A is RANK x RANK)
SIG_K     = 32          # singular values kept in the signature
D_Z       = 16
EPS_T     = 0.05
print('device', DEV, '| model', MODEL_DIR)

In [ ]:
tj = json.load(open(os.path.join(MODEL_DIR, 'tokenizer.json')))
inv_vocab = {i: t for t, i in tj['model']['vocab'].items()}
def decode(ids):
    return ''.join(inv_vocab.get(int(i), '?') for i in ids).replace('\u2581', ' ') \
             .replace('\u0120', ' ').replace('\u010a', '\n')

CFG = json.load(open(os.path.join(MODEL_DIR, 'config.json')))
H, KV = CFG['num_attention_heads'], CFG.get('num_key_value_heads', CFG['num_attention_heads'])
D = CFG['hidden_size']; HD = CFG.get('head_dim', D // H); FF = CFG['intermediate_size']
EPS = CFG.get('rms_norm_eps', 1e-5)
rp = CFG.get('rope_parameters') or {}
THETA = CFG.get('rope_theta', rp.get('rope_theta', 10000))
V = CFG['vocab_size']; MASK_ID = V
PROJ = ('self_attn.q_proj', 'self_attn.k_proj', 'self_attn.v_proj', 'self_attn.o_proj',
        'mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj')

def rotate_half(x):
    d = x.shape[-1] // 2
    return torch.cat([-x[..., d:], x[..., :d]], dim=-1)
def rope(x, pos):
    inv = 1.0 / (THETA ** (torch.arange(0, HD, 2, device=DEV).float() / HD))
    fr = pos[:, None].float() * inv[None, :]
    cos = torch.cat([fr.cos(), fr.cos()], -1)[None, None]
    sin = torch.cat([fr.sin(), fr.sin()], -1)[None, None]
    return x * cos + rotate_half(x) * sin
def rms(x, w): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + EPS) * w

def llama_forward(w, idx, L, causal=True, mask_row=None):
    """Parallel forward for training/eval. Logits over V real tokens (tied head)."""
    B, T = idx.shape
    pos = torch.arange(T, device=DEV)
    E = w['model.embed_tokens.weight']
    x = E[idx.clamp_max(V - 1)]
    if mask_row is not None:
        x = torch.where((idx == MASK_ID)[..., None], mask_row, x)
    bias = torch.full((T, T), float('-inf'), device=DEV).triu(1)[None, None] if causal else None
    for l in range(L):
        p = lambda n: w[f'model.layers.{l}.{n}.weight']
        h = rms(x, p('input_layernorm'))
        q = (h @ p('self_attn.q_proj').T).view(B, T, H, HD).transpose(1, 2)
        k = (h @ p('self_attn.k_proj').T).view(B, T, KV, HD).transpose(1, 2)
        v = (h @ p('self_attn.v_proj').T).view(B, T, KV, HD).transpose(1, 2)
        q, k = rope(q, pos), rope(k, pos)
        k, v = k.repeat_interleave(H // KV, 1), v.repeat_interleave(H // KV, 1)
        att = (q @ k.transpose(-1, -2)) / (HD ** 0.5)
        if bias is not None: att = att + bias
        o = (att.softmax(-1) @ v).transpose(1, 2).reshape(B, T, D)
        x = x + o @ p('self_attn.o_proj').T
        h2 = rms(x, p('post_attention_layernorm'))
        x = x + (F.silu(h2 @ p('mlp.gate_proj').T) * (h2 @ p('mlp.up_proj').T)) @ p('mlp.down_proj').T
    return rms(x, w['model.norm.weight']) @ E.T

@torch.no_grad()
def generate_ar(w, L, prompt, n, temperature=0.9, top_k=40):
    """KV-cached AR sampling (only to capture the donor's own data)."""
    B = prompt.shape[0]; E = w['model.embed_tokens.weight']
    kc, vc = [None] * L, [None] * L
    def step(tokens, off):
        T = tokens.shape[1]; pos = torch.arange(off, off + T, device=DEV)
        x = E[tokens]
        for l in range(L):
            p = lambda n: w[f'model.layers.{l}.{n}.weight']
            h = rms(x, p('input_layernorm'))
            q = (h @ p('self_attn.q_proj').T).view(B, T, H, HD).transpose(1, 2)
            k = (h @ p('self_attn.k_proj').T).view(B, T, KV, HD).transpose(1, 2)
            v = (h @ p('self_attn.v_proj').T).view(B, T, KV, HD).transpose(1, 2)
            q, k = rope(q, pos), rope(k, pos)
            if kc[l] is not None:
                k = torch.cat([kc[l], k], 2); v = torch.cat([vc[l], v], 2)
            kc[l], vc[l] = k, v
            kk, vv = k.repeat_interleave(H // KV, 1), v.repeat_interleave(H // KV, 1)
            att = (q @ kk.transpose(-1, -2)) / (HD ** 0.5)
            Tt = k.shape[2]; kpos = torch.arange(Tt, device=DEV)[None, :]
            att = att + torch.where(kpos <= pos[:, None], 0.0, float('-inf'))
            o = (att.softmax(-1) @ vv).transpose(1, 2).reshape(B, T, D)
            x = x + o @ p('self_attn.o_proj').T
            h2 = rms(x, p('post_attention_layernorm'))
            x = x + (F.silu(h2 @ p('mlp.gate_proj').T) * (h2 @ p('mlp.up_proj').T)) @ p('mlp.down_proj').T
        return (rms(x, w['model.norm.weight']) @ E.T)[:, -1, :]
    logits = step(prompt, 0); out = prompt
    for _ in range(n):
        lo = logits / max(temperature, 1e-6)
        vt, _ = torch.topk(lo, top_k); lo[lo < vt[:, [-1]]] = float('-inf')
        nxt = torch.multinomial(lo.softmax(-1), 1)
        out = torch.cat([out, nxt], 1)
        logits = step(nxt, out.shape[1] - 1)
    return out

supra = {k: v.float().to(DEV) for k, v in load_file(os.path.join(MODEL_DIR, 'model.safetensors')).items()}
L_SUPRA = CFG['num_hidden_layers']
print(f'supra50m: L={L_SUPRA} d={D} V={V}')
s = generate_ar(supra, L_SUPRA, torch.tensor([[1]], device=DEV), 40, temperature=0.7)
print('AR sample (KV-cached):', repr(decode(s[0, 1:])[:160]))

In [ ]:
t0 = time.time(); chunks, need, gb = [], N_GEN + N_HELD, 64
temps = [0.7, 0.9, 1.0]
while sum(c.shape[0] for c in chunks) < need:
    tt = temps[len(chunks) % len(temps)]
    bos = torch.full((gb, 1), 1, dtype=torch.long, device=DEV)
    chunks.append(generate_ar(supra, L_SUPRA, bos, SEQ_LEN, temperature=tt)[:, 1:])
corpus = torch.cat(chunks, 0)[:need]
train_ids, held_ids = corpus[:N_GEN], corpus[N_GEN:]
print(f'corpus from donor: train {tuple(train_ids.shape)} held {tuple(held_ids.shape)} in {time.time()-t0:.0f}s')

In [ ]:
# The zoo: depth-truncated sub-stacks of supra itself. llama_forward(w, idx, L) already
# runs layers 0..L-1 + final norm + tied head, so a truncation is just a smaller L on the
# SAME weight dict -- valid (weaker) AR models, exactly in the target's frame/statistics.
for Lz in TRUNC_DEPTHS:
    ce = F.cross_entropy(llama_forward(supra, held_ids[:8, :-1], Lz).reshape(-1, V),
                         held_ids[:8, 1:].reshape(-1)).item()
    print(f'sub-stack L={Lz:>2}: held AR CE {ce:5.2f}')
print('self-zoo ready (no training needed)')

In [ ]:
def sample_mask_rate(b): return EPS_T + (1.0 - EPS_T) * torch.rand(b, device=DEV)
def forward_mask(x0, t):
    B, L = x0.shape; noise = torch.rand(B, L, device=x0.device); m = noise < t[:, None].expand(B, L)
    empty = ~m.any(dim=1)
    if empty.any(): m[empty.nonzero(as_tuple=True)[0], noise[empty].argmin(dim=1)] = True
    return torch.where(m, torch.full_like(x0, MASK_ID), x0), m
def diffusion_loss(logits, x0, m, t):
    B, L, Vv = logits.shape
    ce = F.cross_entropy(logits.reshape(-1, Vv), x0.reshape(-1), reduction='none').view(B, L)
    return ((ce * m).sum(dim=1) / (t * L)).mean()
@torch.no_grad()
def denoise(fn, ids, frozen, steps=64, temperature=0.0):
    B, L = ids.shape
    for s in range(steps):
        masked = (ids == MASK_ID) & ~frozen; n_left = int(masked.sum().item())
        if n_left == 0: break
        logits = fn(ids)
        probs = (logits / temperature).softmax(-1) if temperature > 0 else logits.softmax(-1)
        pred = torch.multinomial(probs.view(-1, V), 1).view(B, L) if temperature > 0 else probs.argmax(-1)
        conf = probs.max(-1).values.masked_fill(~masked, float('-inf'))
        k = min(max(1, n_left // (steps - s)), n_left)
        ids.view(-1)[conf.view(-1).topk(k).indices] = pred.view(-1)[conf.view(-1).topk(k).indices]
    masked = (ids == MASK_ID) & ~frozen
    if masked.any(): ids = torch.where(masked, fn(ids).argmax(-1), ids)
    return ids
@torch.no_grad()
def masked_ce_at(fn, ids, t_val, n=32):
    x0 = ids[:n]; x_t, m = forward_mask(x0, torch.full((x0.shape[0],), t_val, device=DEV))
    lo = fn(x_t); ce = F.cross_entropy(lo.reshape(-1, V), x0.reshape(-1), reduction='none').view(x0.shape)
    return (ce * m).sum().item() / m.sum().item()
print('diffusion core ready')

In [ ]:
@torch.no_grad()
def svd_cache(w, L):
    """Per layer, per projection: (U_r [o x RANK], V_r [i x RANK], signature)."""
    cache = []
    for l in range(L):
        layer = {}
        for n in PROJ:
            W = w[f'model.layers.{l}.{n}.weight']
            U, S, Vh = torch.linalg.svd(W, full_matrices=False)
            sig = torch.cat([(S / (S.norm() + 1e-9))[:SIG_K],
                             torch.log(W.norm() + 1e-9)[None]])
            layer[n] = (U[:, :RANK].contiguous(), Vh[:RANK].T.contiguous(), sig)
        cache.append(layer)
    return cache

SIG_DIM = (SIG_K + 1) * len(PROJ)        # signature = concat of all 7 projections' (spectrum+lognorm)

class TranslatorC(nn.Module):
    def __init__(self, d_z=D_Z, r=RANK, h=128):
        super().__init__(); self.r = r
        self.enc = nn.Sequential(nn.Linear(SIG_DIM + 1, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, d_z))
        self.A = nn.ParameterDict({n.replace('.', '_'): nn.Parameter(torch.zeros(r * r, d_z)) for n in PROJ})
        self.Mg = nn.Parameter(torch.zeros(2, d_z))     # per-norm scalar corrections (frame-free)
        self.mask_logits = nn.Parameter(torch.zeros(V)) # frame-free [MASK] = softmax-combo of own rows
    def mask_row(self, w): return self.mask_logits.softmax(0) @ w['model.embed_tokens.weight']
    def emit(self, w, L, cache):
        out = dict(w)
        for l in range(L):
            sig = torch.cat([cache[l][n][2] for n in PROJ])
            z = self.enc(torch.cat([sig, sig.new_tensor([0.0 if L == 1 else l / (L - 1)])]))
            for n in PROJ:
                U, Vr, _ = cache[l][n]
                A = (self.A[n.replace('.', '_')] @ z).view(self.r, self.r)
                out[f'model.layers.{l}.{n}.weight'] = w[f'model.layers.{l}.{n}.weight'] + U @ A @ Vr.T
            dg = (self.Mg @ z)
            out[f'model.layers.{l}.input_layernorm.weight'] = w[f'model.layers.{l}.input_layernorm.weight'] * (1 + dg[0])
            out[f'model.layers.{l}.post_attention_layernorm.weight'] = w[f'model.layers.{l}.post_attention_layernorm.weight'] * (1 + dg[1])
        return out

t0 = time.time()
supra_cache = svd_cache(supra, L_SUPRA)      # one cache; truncations slice it
C = TranslatorC().to(DEV)
print(f'SVD cached + C built ({sum(p.numel() for p in C.parameters())} params, {time.time()-t0:.0f}s)')

In [ ]:
opt = torch.optim.AdamW(C.parameters(), lr=3e-4)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 100))
ema, t0 = None, time.time()
for step in range(1, C_STEPS + 1):
    Lz = TRUNC_DEPTHS[step % len(TRUNC_DEPTHS)]
    x0 = train_ids[torch.randint(0, train_ids.shape[0], (C_BS,), device=DEV)]
    t = sample_mask_rate(C_BS); x_t, m = forward_mask(x0, t)
    wB = C.emit(supra, Lz, supra_cache)
    loss = diffusion_loss(llama_forward(wB, x_t, Lz, causal=False, mask_row=C.mask_row(supra)), x0, m, t)
    opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(C.parameters(), 1.0); opt.step(); sched.step()
    ema = loss.item() if ema is None else 0.98 * ema + 0.02 * loss.item()
    if step % 500 == 0 or step == 1:
        with torch.no_grad():
            fn = lambda i: llama_forward(C.emit(supra, 4, supra_cache), i, 4, causal=False, mask_row=C.mask_row(supra))
            hce = masked_ce_at(fn, held_ids, 0.5, n=16)
        print(f'step {step:>5}  train-ELBO(ema) {ema:6.3f}  sub-stack-L4 masked-CE@0.5 {hce:5.2f}  ({time.time()-t0:.0f}s)')
print('C trained — B was never trained, only C was')

In [ ]:
with torch.no_grad():
    wB = C.emit(supra, L_SUPRA, supra_cache); mrow = C.mask_row(supra)
fnB = lambda i: llama_forward(wB, i, L_SUPRA, causal=False, mask_row=mrow)
mean_row = supra['model.embed_tokens.weight'].mean(0)
fn0 = lambda i: llama_forward(supra, i, L_SUPRA, causal=False, mask_row=mean_row)   # floor: no translation

ar = F.cross_entropy(llama_forward(supra, held_ids[:16, :-1], L_SUPRA).reshape(-1, V), held_ids[:16, 1:].reshape(-1)).item()
print(f'uniform ln(V) = {math.log(V):.2f} | donor AR next-token CE (held) = {ar:.2f}')
print('\nheld masked-CE:    floor(raw bidir)   B*=C(supra)   [lower=better]')
for tv in (0.3, 0.5, 0.7, 0.9):
    print(f'  t={tv}:   {masked_ce_at(fn0, held_ids, tv):7.2f}      {masked_ce_at(fnB, held_ids, tv):7.2f}')

x0 = held_ids[:8]; x_c, m = forward_mask(x0, torch.full((x0.shape[0],), 0.25, device=DEV))
for name, fn in (('floor', fn0), ('B*=C(supra)', fnB)):
    rec = denoise(fn, x_c.clone(), ~m, steps=16); acc = ((rec == x0) & m).sum().item() / m.sum().item()
    print(f'\nreconstruction ({name}): token accuracy {acc:.1%}')
    if name.startswith('B'):
        print('  original :', repr(decode(x0[0, :48]))); print('  recovered:', repr(decode(rec[0, :48])))

ids = torch.full((2, 128), MASK_ID, dtype=torch.long, device=DEV); frozen = torch.zeros(2, 128, dtype=torch.bool, device=DEV)
gen = denoise(fnB, ids, frozen, steps=64, temperature=0.7)
for b in range(2): print(f'\nB* parallel sample {b}:', repr(decode(gen[b])[:300]))

# extrapolation probe: was full-depth composition the hard part? deltas applied only to
# blocks 0..9 (seen in training contexts) vs all 12
with torch.no_grad():
    wB10 = dict(supra)
    wTmp = C.emit(supra, 10, supra_cache)
    for k in wTmp:
        wB10[k] = wTmp[k]
fn10 = lambda i: llama_forward(wB10, i, L_SUPRA, causal=False, mask_row=mrow)
print(f'deltas on blocks 0..9 only (full L=12 run) masked-CE@0.5: {masked_ce_at(fn10, held_ids, 0.5):.2f}')

with torch.no_grad():
    perm = torch.randperm(L_SUPRA).tolist()
    wB_mm = C.emit(supra, L_SUPRA, [supra_cache[p] for p in perm])
fn_mm = lambda i: llama_forward(wB_mm, i, L_SUPRA, causal=False, mask_row=mrow)
print(f'\ncontrol — shuffled-signature emit masked-CE@0.5: {masked_ce_at(fn_mm, held_ids, 0.5):.2f} '
      f'(vs B* {masked_ce_at(fnB, held_ids, 0.5):.2f}; equal ⇒ C ignores signatures)')

In [ ]:
out_sd = {k: v.detach().cpu().contiguous() for k, v in wB.items()}
out_sd['mercury.mask_embedding'] = mrow.detach().cpu().contiguous()
path = '/kaggle/working/supra_mercury2_by_C.safetensors' if os.path.isdir('/kaggle/working') else 'supra_mercury2_by_C.safetensors'
save_file(out_sd, path)
print('saved B* = C(supra50m) ->', path, f'({os.path.getsize(path)/1e6:.0f} MB)')

## How to read
- **Floor vs B\***: the floor is the untranslated donor forced bidirectional. If `B*=C(supra)`
  now beats it (it did NOT in run 1), the SVD-frame deltas transferred from the zoo and the
  translator alone moved supra toward a denoiser — **with zero training of B**.
- **Shuffled-signature control**: in run 1 it matched B* (C ignored signatures, frame noise).
  A gap now ⇒ C genuinely reads each block's spectrum and the delta lands in-frame.
- **Honest expectation**: C extrapolates from depth-2..4 fresh donors to a trained L=12 model;
  per phase-10 this works for refinement-style interiors. Partial floor→ceiling gap closure
  (ceiling = the BASELINE `supra_to_mercury.ipynb` finetune) is the headline; B* is never trained.